In [ ]:
import os
import re
import math

import numpy as np
import pandas as pd
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests

import cfospy
import costest

In [ ]:
def rad2ph(rad):
    if math.isnan(rad):
        return np.nan
    else:
        return (round((2*np.pi+rad)*180/np.pi*24/360, 1),  round((rad)*180/np.pi*24/360, 1))[rad>=0]

def ct_number_shift(col_name, shift=48, prefix="CT"):
    if col_name.startswith(prefix) and "_" in col_name:
        parts = col_name.split("_")
        try:
            number = int(parts[0][len(prefix):])
            new_number = number + shift
            return f"{prefix}{new_number}_{parts[1]}"
        except ValueError:
            return col_name
    return col_name

def rename_column(col):
    col = str(col)
    m = re.match(r"(CT)(\d+)_(\d+)", col)
    if m:
        base, num, rep = m.groups()
        num = int(num)
        if num < 48:
            return f"1st_{base}{num}_{rep}"
        else:
            return f"2nd_{base}{num - 48}_{rep}"
    return col

In [ ]:
src = "/path/to/source_dir"
dst = "/path/to/output_dir"
savedir = os.path.join(dst, "cfos_app")

In [ ]:
# Load atlas data
rdir = os.path.join(src, "CUBIC_R_atlas_ver5")
vx = 50
ca = cfospy.analysis.read_atlas_data(rdir, vx)
print(f"Number of all regions: {len(ca.ID_all)}")

In [ ]:
# Analytic cosinor test (2 series concatenated)

cos_dir = os.path.join(src, "cos_results")
os.makedirs(cos_dir, exist_ok=True)

typev = "count"
exps = ["1st", "2nd"]

# Load summary data from 1st/2nd series
file_name = f"region_cell_ai_fpr0.5_{typev}_summary"
path_r1 = os.path.join(savedir, exps[0], f"{file_name}_{exps[0]}.csv")
path_r2 = os.path.join(savedir, exps[1], f"{file_name}_{exps[1]}.csv")

df_r1 = pd.read_csv(path_r1)
df_r2 = pd.read_csv(path_r2)
df_r2.columns = [ct_number_shift(c, shift=48) for c in df_r2.columns]
df = pd.merge(df_r1, df_r2, on="id", how="inner")

avg_list = []
se_list = []
for i in range(df.shape[0]):
    vals = df.iloc[i, 1:].to_numpy(dtype=float)
    if vals.size != 144:  # 6 reps × 12 CT × 2 series = 144
        raise ValueError(f"Unexpected column count ({vals.size}).")
    tbl = vals.reshape(24, 6)
    avg = tbl.mean(axis=1)
    se = tbl.std(axis=1, ddof=1) / np.sqrt(tbl.shape[1])
    avg_list.append(avg)
    se_list.append(se)

# Batch analysis over multiple time series with corresponding SEM
data_matrix = np.array(avg_list)
se_matrix = np.array(se_list)
results = batch_costest(data_matrix, 6, se_matrix)

mc, mc_ph, p_org, p_sem_adj = results[:, 0], results[:, 1], results[:, 2], results[:, 3]
nan_id = np.isnan(p_sem_adj)
nonan_id = np.where(nan_id == False)[0]

phase_li = list(map(rad2ph, mc_ph))
acronyms = [ca.df_allen[ca.df_allen["ID"] == rID]["acronym"].iloc[0] for rID in df["id"]]
node_names = [ca.df_allen[ca.df_allen["ID"] == rID]["node_name"].iloc[0] for rID in df["id"]]
per_li = np.ones(len(df)) * 24

cos_v_df = pd.DataFrame({
    "id": df["id"].tolist(),
    "acronym": acronyms,
    "node_name": node_names,
    "ADJ.P": p_sem_adj,
    "PER": per_li,
    "Ph": mc_ph,
    "LAG": phase_li,
    "max_corr": mc,
})

# Benjamini-Hochberg FDR on non-NaN p-values
q_bh_np = np.ones(len(p_sem_adj))
_, q_bh_li, _, _ = multipletests(p_sem_adj[nonan_id], method="fdr_bh")
q_bh_np[nonan_id] = q_bh_li
cos_v_df.insert(3, "BH.Q", q_bh_np)

# Append original time-series columns (from col 9 onward)
cos_v_df = pd.concat([cos_v_df, df.iloc[:, 1:]], axis=1)

# Sort by adjusted p-value
cos_v_df = cos_v_df.sort_values("ADJ.P")

# Construct filename
out_name = f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5.csv"
cos_file = os.path.join(cos_dir, out_name)

# Save CSV using a context manager
with open(cos_file, "w", newline="") as f:
    cos_v_df.to_csv(f)

# Report significant count under BH.Q threshold
th = 0.1
print(alpha)
print(len(cos_v_df[cos_v_df["BH.Q"] < th]))

cos_v_df

In [ ]:
# Output for other analysis patterns
other_dir = os.path.join(src, 'other_cos_results')
os.makedirs(other_dir, exist_ok=True)

In [ ]:
# Rename columns to add 1st_/2nd_ prefixes and adjust CT numbers

cos_file = os.path.join(cos_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5.csv")
df = pd.read_csv(cos_file)

df.columns = [rename_column(c) for c in df.columns]

renamed_path = os.path.join(cos_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_renamed.csv")
df.to_csv(renamed_path, index=False)

print("Column names updated and saved as:")
print(os.path.basename(renamed_path))

In [ ]:
# Analytic cosinor test for each experiment (2 cycles: 48h)

typev = "count"
exps = ["1st", "2nd"]

ct_li_2d = np.arange(0, 48, 4)
sample_ids = np.arange(1, 7, 1)

cos_file = os.path.join(cos_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_renamed.csv")
print(os.path.basename(cos_file))

df = pd.read_csv(cos_file)
IDs = df["id"].tolist()
print(f"{len(IDs)} regions")

for exp in exps:
    ct_mean_li = []
    ct_se_li = []
    for ct in ct_li_2d:
        print(f"{exp}_CT{ct}")
        cols = [c for c in df.columns if c.startswith(f"{exp}_CT{ct}_")]
        df_mean = df[cols].mean(axis=1)
        df_sem  = df[cols].std(axis=1) / np.sqrt(len(sample_ids))
    
        ct_mean_li.append(df_mean)
        ct_se_li.append(df_sem)
    
    df_r_mean = pd.concat(ct_mean_li, axis=1)
    df_r_se = pd.concat(ct_se_li, axis=1)
    print(df_r_mean)
    
    
    results = batch_costest(df_r_mean.to_numpy(), len(sample_ids), df_r_se.to_numpy())
    mc        = results[:, 0]
    mc_ph     = results[:, 1]
    p_org     = results[:, 2]
    p_sem_adj = results[:, 3]
    
    nan_id = np.isnan(p_sem_adj)
    nonan_id = np.where(~nan_id)[0]
    
    phase_li = list(map(rad2ph, mc_ph))
    per_li = np.ones(len(df)) * 24
    
    cos_v_df = pd.DataFrame({
        "id": df["id"],
        "acronym": df["acronym"],
        "node_name": df["node_name"],
        "ADJ.P": p_sem_adj,
        "PER": per_li,
        "Ph": mc_ph,
        "LAG": phase_li,
        "max_corr": mc,
    })
    
    q_bh_np = np.ones(len(p_sem_adj))
    _, q_bh_li, _, _ =  multipletests(p_sem_adj[nonan_id],  method="fdr_bh")
    
    q_bh_np[nonan_id] = q_bh_li
    cos_v_df.insert(3, "BH.Q", q_bh_np)

    exp_cols = [c for c in df.columns if c.startswith(f"{exp}_CT")]
    cos_v_df = pd.concat([cos_v_df, df[exp_cols]], axis=1)
    
    cos_v_df = cos_v_df.sort_values(by=["BH.Q","ADJ.P"])
    print(cos_v_df)
    
    cos_v_df.to_csv(os.path.join(other_dir, f"cos.cell_{typev}_{exp}_small_ai_fpr0.5.csv"), index=False)
    
    print()
    print(f"Each experiment ({exp}):")
    print(f"p < 0.05: {len(cos_v_df[cos_v_df['ADJ.P'] < 0.05])} regions")
    print(f"FDR < 0.1: {len(cos_v_df[cos_v_df['BH.Q'] < 0.1])} regions")
    print()

In [ ]:
# Combine p-values (2 exps)
typev = "count"
exps = ["1st", "2nd"]

df1 = pd.read_csv(os.path.join(other_dir, f"cos.cell_{typev}_{exps[0]}_small_ai_fpr0.5.csv"))
df1 = df1.sort_values("id").reset_index(drop=True)
df2 = pd.read_csv(os.path.join(other_dir, f"cos.cell_{typev}_{exps[1]}_small_ai_fpr0.5.csv"))
df2 = df2.sort_values("id").reset_index(drop=True)

if not (df1["id"].is_unique and df2["id"].is_unique):
    raise ValueError("Duplicate id detected in df1 or df2")
    
if not df1["id"].equals(df2["id"]):
    raise ValueError("ID columns do not match between df1 and df2")

com_pvals = []
for n in range(len(df1)):
    p1 = df1["ADJ.P"].iloc[n]
    p2 = df2["ADJ.P"].iloc[n]
    if not np.isfinite(p1) or not np.isfinite(p2) or p1 <= 0 or p2 <= 0 or p1 > 1 or p2 > 1:
        com_pvals.append(np.nan)
    else:
        _, p_fisher = combine_pvalues([p1, p2], method="fisher")
        com_pvals.append(p_fisher)

print(len(com_pvals))

com_pvals_np = np.array(com_pvals)
nan_id = np.isnan(com_pvals_np)
nonan_id = np.where(nan_id==False)[0]

q_bh_np = np.ones(len(com_pvals_np))
if len(nonan_id) > 0:
    _, q_bh_li, _, _ =  multipletests((com_pvals_np)[nonan_id],  method="fdr_bh")
    q_bh_np[nonan_id] = q_bh_li

if not (df1["id"].is_unique and df2["id"].is_unique):
    raise ValueError("Duplicate id detected in df1 or df2")
if set(df1["id"]) != set(df2["id"]):
    raise ValueError("id sets differ between df1 and df2")
    
com_df = df1.iloc[:, :6].copy()
com_df = com_df.rename(columns={"ADJ.P": "COM.P"})
com_df["COM.P"] = com_pvals_np
com_df["BH.Q"] = q_bh_np

cols_stat = ["Ph", "LAG", "max_corr"]
df1_stats = df1[["id"] + cols_stat].rename(columns={c: f"{c}_{exps[0]}" for c in cols_stat})
df2_stats = df2[["id"] + cols_stat].rename(columns={c: f"{c}_{exps[1]}" for c in cols_stat})

df1_ct = df1[["id"] + [c for c in df1.columns if c.startswith(f"{exps[0]}_CT")]]
df2_ct = df2[["id"] + [c for c in df2.columns if c.startswith(f"{exps[1]}_CT")]]

com_df = (
    com_df
    .merge(df1_stats, on="id", how="left")
    .merge(df2_stats, on="id", how="left")
    .merge(df1_ct,   on="id", how="left")
    .merge(df2_ct,   on="id", how="left")
)

com_df = com_df.sort_values(by=["BH.Q", "COM.P"])
print(com_df)

com_file = os.path.join(other_dir, f"cos.cell_{typev}_{exps[0]}{exps[1]}_small_ai_fpr0.5_2cycle_p_combined.csv")
com_df.to_csv(com_file, index=False)

print()
print("2 cycles x 2 (p combined):")
print(f"p < 0.05: {len(com_df[com_df['COM.P'] < 0.05])} regions")
print(f"FDR < 0.1: {len(com_df[com_df['BH.Q'] < 0.1])} regions")
print(f"Saved {com_file}")

In [ ]:
# Prepare for n-combined test (2×2 cycles)
typev = "count"
exps = ["1st", "2nd"]

ct_li_2d = np.arange(0, 48, 4)
sample_ids = np.arange(1, 7, 1)

cos_file = os.path.join(cos_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_renamed.csv")
print(os.path.basename(cos_file))

df = pd.read_csv(cos_file)
print(f"{len(df)} regions")

df_sum = df[["id", "acronym", "node_name"]].copy()

cols1 = [c for c in df.columns if c.startswith(f"{exps[0]}_CT")]
cols2 = [c for c in df.columns if c.startswith(f"{exps[1]}_CT")]

if len(cols1) != len(cols2):
    raise ValueError(f"CT{ct}: number of columns mismatch (1st={len(cols1)}, 2nd={len(cols2)})")

summed = df[cols1].to_numpy() + df[cols2].to_numpy()

sum_cols = [c.replace("1st_", "sum_") for c in cols1]
df_sum = pd.concat([df_sum, pd.DataFrame(summed, columns=sum_cols)], axis=1)
print(df_sum)

sum_file = os.path.join(other_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_2cycle_sum.csv")
df_sum.to_csv(sum_file, index=False)

print(f"Saved {sum_file}")

In [ ]:
# Analytic cosinor test (n-combined 2×2)
typev = "count"

ct_li_2d= np.arange(0, 48, 4)
sample_ids = np.arange(1, 7, 1)

com_file = os.path.join(other_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_2cycle_sum.csv")
print(os.path.basename(com_file))

df = pd.read_csv(com_file)
IDs = df["id"].tolist()
print(f"{len(IDs)} regions")

ct_mean_li = []
ct_se_li = []
for i, ct in enumerate(ct_li_2d):
    print(f"sum_CT{ct}")
    cols = [c for c in df.columns if c.startswith(f"sum_CT{ct}_")]
    df_mean = df[cols].mean(axis=1)
    df_sem  = df[cols].std(axis=1) / np.sqrt(len(sample_ids))

    ct_mean_li.append(df_mean)
    ct_se_li.append(df_sem)

df_r_mean = pd.concat(ct_mean_li, axis=1)
df_r_se = pd.concat(ct_se_li, axis=1)
print(df_r_mean)


results = batch_costest(df_r_mean.to_numpy(), len(sample_ids), df_r_se.to_numpy())
mc        = results[:, 0]
mc_ph     = results[:, 1]
p_org     = results[:, 2]
p_sem_adj = results[:, 3]

nan_id = np.isnan(p_sem_adj)
nonan_id = np.where(nan_id==False)[0]

phase_li = list(map(rad2ph, mc_ph))
per_li = np.ones(len(df)) * 24

cos_v_df = pd.DataFrame({
    "id": df["id"],
    "acronym": df["acronym"],
    "node_name": df["node_name"],
    "ADJ.P": p_sem_adj,
    "PER": per_li,
    "Ph": mc_ph,
    "LAG": phase_li,
    "max_corr": mc,
})

q_bh_np = np.ones(len(p_sem_adj))
_, q_bh_li, _, _ =  multipletests(p_sem_adj[nonan_id],  method="fdr_bh")

q_bh_np[nonan_id] = q_bh_li
cos_v_df.insert(3, "BH.Q", q_bh_np)

ct_cols = [c for c in df.columns if c.startswith(f"sum_CT")]
cos_v_df = pd.concat([cos_v_df, df[ct_cols]], axis=1)

cos_v_df = cos_v_df.sort_values(by=["BH.Q","ADJ.P"])
print(cos_v_df)

cos_v_df.to_csv(os.path.join(other_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_2cycle_n_combined.csv"), index=False)

print()
print("2 x 2 cycles (n combined):")
print(f"p < 0.05: {len(cos_v_df[cos_v_df['ADJ.P'] < 0.05])} regions")
print(f"FDR < 0.1: {len(cos_v_df[cos_v_df['BH.Q'] < 0.1])} regions")

In [ ]:
# Analytic cosinor test (each 24h cycle)
typev = "count"
exps = ["1st", "2nd"]
cycles = [1, 2]

ct_li_1d = np.arange(0, 24, 4)
sample_ids  = np.arange(1, 7, 1)

cos_file = os.path.join(cos_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_renamed.csv")
print(os.path.basename(cos_file))

df = pd.read_csv(cos_file)
IDs = df["id"].tolist()
print(f"{len(IDs)} regions")

for exp in exps:
    for cycle in cycles:
        print(f"{exp} experiment / cycle {cycle}")
        offset = (cycle - 1) * 24

        ct_mean_li = []
        ct_se_li = []
        for ct in ct_li_1d:
            org_ct = ct + offset
            print(f"{exp}_CT{org_ct}")
            cols = [c for c in df.columns if c.startswith(f"{exp}_CT{org_ct}_")]
            df_mean = df[cols].mean(axis=1)
            df_sem  = df[cols].std(axis=1) / np.sqrt(len(sample_ids))
            ct_mean_li.append(df_mean)
            ct_se_li.append(df_sem)
        
        df_r_mean = pd.concat(ct_mean_li, axis=1)
        df_r_se   = pd.concat(ct_se_li, axis=1)
        print(df_r_mean)
        
        results   = batch_costest(df_r_mean.to_numpy(), len(sample_ids), df_r_se.to_numpy())
        mc        = results[:, 0]
        mc_ph     = results[:, 1]
        p_org     = results[:, 2]
        p_sem_adj = results[:, 3]
        
        nan_id   = np.isnan(p_sem_adj)
        nonan_id = np.where(~nan_id)[0]
        
        phase_li = list(map(rad2ph, mc_ph))
        per_li   = np.ones(len(df)) * 24
        
        cos_v_df = pd.DataFrame({
            "id": df["id"],
            "acronym": df["acronym"],
            "node_name": df["node_name"],
            "ADJ.P": p_sem_adj,
            "PER": per_li,
            "Ph": mc_ph,
            "LAG": phase_li,
            "max_corr": mc,
        })
        
        q_bh_np = np.ones(len(p_sem_adj))
        _, q_bh_li, _, _ = multipletests(p_sem_adj[nonan_id], method="fdr_bh")
        
        q_bh_np[nonan_id] = q_bh_li
        cos_v_df.insert(3, "BH.Q", q_bh_np)

        ct_this = [ct + offset for ct in ct_li_1d]
        exp_cols = [c for c in df.columns if any(c.startswith(f"{exp}_CT{t}_") for t in ct_this)]
        cos_v_df = pd.concat([cos_v_df, df[exp_cols]], axis=1)
        
        cos_v_df = cos_v_df.sort_values(by=["BH.Q", "ADJ.P"])
        print(cos_v_df)
        
        cos_v_df.to_csv(os.path.join(other_dir, f"cos.cell_{typev}_{exp}_c{cycle}_small_ai_fpr0.5.csv"), index=False)
        
        print()
        print(f"Each cycle ({exp} experiment, cycle {cycle}):")
        print(f"p < 0.05: {len(cos_v_df[cos_v_df['ADJ.P'] < 0.05])} regions")
        print(f"FDR < 0.1: {len(cos_v_df[cos_v_df['BH.Q'] < 0.1])} regions")
        print()

In [ ]:
# Combine p-values (1×4 cycles)
typev = "count"
exps = ["1st", "2nd"]
cycles = [1, 2]

df1_c1 = pd.read_csv(os.path.join(other_dir, f"cos.cell_{typev}_{exps[0]}_c{cycles[0]}_small_ai_fpr0.5.csv"))
df1_c1 = df1_c1.sort_values("id").reset_index(drop=True)

df1_c2 = pd.read_csv(os.path.join(other_dir, f"cos.cell_{typev}_{exps[0]}_c{cycles[1]}_small_ai_fpr0.5.csv"))
df1_c2 = df1_c2.sort_values("id").reset_index(drop=True)

df2_c1 = pd.read_csv(os.path.join(other_dir, f"cos.cell_{typev}_{exps[1]}_c{cycles[0]}_small_ai_fpr0.5.csv"))
df2_c1 = df2_c1.sort_values("id").reset_index(drop=True)

df2_c2 = pd.read_csv(os.path.join(other_dir, f"cos.cell_{typev}_{exps[1]}_c{cycles[1]}_small_ai_fpr0.5.csv"))
df2_c2 = df2_c2.sort_values("id").reset_index(drop=True)

for dname, d in [("df1_c1", df1_c1), ("df1_c2", df1_c2), ("df2_c1", df2_c1), ("df2_c2", df2_c2)]:
    if not d["id"].is_unique:
        raise ValueError(f"Duplicate id detected in {dname}")

if not (df1_c1["id"].equals(df1_c2["id"]) and df1_c1["id"].equals(df2_c1["id"]) and df1_c1["id"].equals(df2_c2["id"])):
    raise ValueError("ID columns do not match across the four dataframes")

p1 = pd.to_numeric(df1_c1["ADJ.P"], errors="coerce").to_numpy()
p2 = pd.to_numeric(df1_c2["ADJ.P"], errors="coerce").to_numpy()
p3 = pd.to_numeric(df2_c1["ADJ.P"], errors="coerce").to_numpy()
p4 = pd.to_numeric(df2_c2["ADJ.P"], errors="coerce").to_numpy()

com_pvals = []
for a, b, c, d in zip(p1, p2, p3, p4):
    pvals = np.array([a, b, c, d], dtype=float)
    if np.any(~np.isfinite(pvals)) or np.any(pvals <= 0) or np.any(pvals > 1):
        com_pvals.append(np.nan)
    else:
        _, p_fisher = combine_pvalues(pvals, method="fisher")
        com_pvals.append(p_fisher)

com_pvals_np = np.array(com_pvals)
nonan_id = np.flatnonzero(~np.isnan(com_pvals_np))

q_bh_np = np.ones(len(com_pvals_np))
if len(nonan_id) > 0:
    _, q_bh_li, _, _ = multipletests(com_pvals_np[nonan_id], method="fdr_bh")
    q_bh_np[nonan_id] = q_bh_li

com_df = df1_c1.iloc[:, :6].copy()
com_df = com_df.rename(columns={"ADJ.P": "COM.P"})
com_df["COM.P"] = com_pvals_np
com_df["BH.Q"]  = q_bh_np

cols_stat = ["Ph", "LAG", "max_corr"]

def stats_rename(df, exp, cyc):
    return df[["id"] + cols_stat].rename(columns={c: f"{c}_{exp}_c{cyc}" for c in cols_stat})

com_df = (com_df
          .merge(stats_rename(df1_c1, exps[0], cycles[0]), on="id", how="left")
          .merge(stats_rename(df1_c2, exps[0], cycles[1]), on="id", how="left")
          .merge(stats_rename(df2_c1, exps[1], cycles[0]), on="id", how="left")
          .merge(stats_rename(df2_c2, exps[1], cycles[1]), on="id", how="left"))

com_df = (com_df
          .merge(df1_c1[["id"] + [c for c in df1_c1.columns if c.startswith(f"{exps[0]}_CT")]], on="id", how="left")
          .merge(df1_c2[["id"] + [c for c in df1_c2.columns if c.startswith(f"{exps[0]}_CT")]], on="id", how="left")
          .merge(df2_c1[["id"] + [c for c in df2_c1.columns if c.startswith(f"{exps[1]}_CT")]], on="id", how="left")
          .merge(df2_c2[["id"] + [c for c in df2_c2.columns if c.startswith(f"{exps[1]}_CT")]], on="id", how="left")
)


com_df = com_df.sort_values(by=["BH.Q", "COM.P"])
print(com_df)

com_file = os.path.join(other_dir, f"cos.cell_{typev}_{exps[0]}{exps[1]}_small_ai_fpr0.5_1cycle_p_combined.csv")
com_df.to_csv(com_file, index=False)

print()
print("1 cycle x 4 (p combined):")
print(f"p < 0.05: {len(com_df[com_df['COM.P'] < 0.05])} regions")
print(f"FDR < 0.1: {len(com_df[com_df['BH.Q'] < 0.1])} regions")
print(f"Saved {com_file}")

In [ ]:
# Prepare for n-combined test (1×4 cycles)
typev  = "count"
exps   = ["1st", "2nd"]
cycles = [1, 2]

ct_li_1d   = np.arange(0, 24, 4)
sample_ids = np.arange(1, 7, 1)

cos_file = os.path.join(cos_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_renamed.csv")
print(os.path.basename(cos_file))

df = pd.read_csv(cos_file)
print(f"{len(df)} regions")

df_sum = df[["id", "acronym", "node_name"]].copy()

cols_1st_c1 = [c for t in ct_li_1d for c in df.columns if c.startswith(f"1st_CT{t}_")]
cols_1st_c2 = [c for t in ct_li_1d for c in df.columns if c.startswith(f"1st_CT{t+24}_")]
cols_2nd_c1 = [c for t in ct_li_1d for c in df.columns if c.startswith(f"2nd_CT{t}_")]
cols_2nd_c2 = [c for t in ct_li_1d for c in df.columns if c.startswith(f"2nd_CT{t+24}_")]

n1, n2, n3, n4 = map(len, (cols_1st_c1, cols_1st_c2, cols_2nd_c1, cols_2nd_c2))
if not (n1 == n2 == n3 == n4):
    raise ValueError(f"column count mismatch: 1st_c1={n1}, 1st_c2={n2}, 2nd_c1={n3}, 2nd_c2={n4}")

summed = (
    df[cols_1st_c1].to_numpy()
  + df[cols_1st_c2].to_numpy()
  + df[cols_2nd_c1].to_numpy()
  + df[cols_2nd_c2].to_numpy()
)

sum_cols = [c.replace("1st_", "sum_") for c in cols_1st_c1]

df_sum = pd.concat([df_sum, pd.DataFrame(summed, columns=sum_cols)], axis=1)
print(df_sum)

sum_file = os.path.join(other_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_1cycle_sum.csv")
df_sum.to_csv(sum_file, index=False)
print(f"Saved {sum_file}")

In [ ]:
# Analytic cosinor test (n-combined 1×4)
typev = "count"

ct_li_1d= np.arange(0, 24, 4)
sample_ids = np.arange(1, 7, 1)

com_file = os.path.join(other_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_1cycle_sum.csv")
print(os.path.basename(com_file))

df = pd.read_csv(com_file)
IDs = df["id"].tolist()
print(f"{len(IDs)} regions")

ct_mean_li = []
ct_se_li = []
for i, ct in enumerate(ct_li_1d):
    print(f"sum_CT{ct}")
    cols = [c for c in df.columns if c.startswith(f"sum_CT{ct}_")]
    df_mean = df[cols].mean(axis=1)
    df_sem  = df[cols].std(axis=1) / np.sqrt(len(sample_ids))

    ct_mean_li.append(df_mean)
    ct_se_li.append(df_sem)

df_r_mean = pd.concat(ct_mean_li, axis=1)
df_r_se = pd.concat(ct_se_li, axis=1)
print(df_r_mean)


results = batch_costest(df_r_mean.to_numpy(), len(sample_ids), df_r_se.to_numpy())
mc        = results[:, 0]
mc_ph     = results[:, 1]
p_org     = results[:, 2]
p_sem_adj = results[:, 3]

nan_id = np.isnan(p_sem_adj)
nonan_id = np.where(nan_id==False)[0]

phase_li = list(map(rad2ph, mc_ph))
per_li = np.ones(len(df)) * 24

cos_v_df = pd.DataFrame({
    "id": df["id"],
    "acronym": df["acronym"],
    "node_name": df["node_name"],
    "ADJ.P": p_sem_adj,
    "PER": per_li,
    "Ph": mc_ph,
    "LAG": phase_li,
    "max_corr": mc,
})

q_bh_np = np.ones(len(p_sem_adj))
_, q_bh_li, _, _ =  multipletests(p_sem_adj[nonan_id],  method="fdr_bh")

q_bh_np[nonan_id] = q_bh_li
cos_v_df.insert(3, "BH.Q", q_bh_np)

ct_cols = [c for c in df.columns if c.startswith(f"sum_CT")]
cos_v_df = pd.concat([cos_v_df, df[ct_cols]], axis=1)

cos_v_df = cos_v_df.sort_values(by=["BH.Q","ADJ.P"])
print(cos_v_df)

cos_v_df.to_csv(os.path.join(other_dir, f"cos.cell_{typev}_1st2nd_small_ai_fpr0.5_1cycle_n_combined.csv"), index=False)

print()
print("4 x 1 cycle (n combined):")
print(f"p < 0.05: {len(cos_v_df[cos_v_df['ADJ.P'] < 0.05])} regions")
print(f"FDR < 0.1: {len(cos_v_df[cos_v_df['BH.Q'] < 0.1])} regions")